# NLP Lecture Notebook: From Transformers to Modern LLMs

Welcome to the practical companion for our lecture on modern NLP. In this notebook, we will implement the core concepts discussed, from tokenization to interacting with local and cloud-based Large Language Models.

## 0. Setup

First, we need to install the necessary Python libraries. We will use:
- `transformers` and `torch` for low-level NLP tasks like tokenization.
- `openai` to interact with the OpenAI API.
- `ollama` to interact with models running locally.

After running the cell below, you may need to restart your runtime environment.

In [3]:
# Use a ! to run shell commands from inside the notebook
!pip install -q transformers torch openai ollama python-dotenv

print("Libraries installed successfully!")


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Libraries installed successfully!


In [5]:
# Load openai key from .env file
from dotenv import load_dotenv

load_dotenv()

True

## 1. Tokenization in Practice

This section corresponds to **Part 3** of the presentation.

As we discussed, models don't see words; they see tokens. Let's make this concept concrete. We will use a tokenizer from the Hugging Face `transformers` library to see how a sentence is broken down into subwords and then converted into numerical IDs. We'll use the classic `gpt2` tokenizer for this example.

In [6]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer. GPT-2 is a great example.
tokenizer = AutoTokenizer.from_pretrained("gpt2")

sentence = "The Transformer architecture is unbelievably powerful."

# 1. Tokenize the sentence into subword strings
tokens = tokenizer.tokenize(sentence)

print(f"Original Sentence: {sentence}")
print("-" * 30)
print(f"Subword Tokens: {tokens}")
print("-" * 30)


# 2. Encode the sentence into integer IDs
# This is what the model actually receives as input
input_ids = tokenizer.encode(sentence)

print(f"Input IDs: {input_ids}")
print("-" * 30)

# You can also decode the IDs back to a string to verify
decoded_sentence = tokenizer.decode(input_ids)
print(f"Decoded Sentence: {decoded_sentence}")

/home/pc/.pyenv/versions/test_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original Sentence: The Transformer architecture is unbelievably powerful.
------------------------------
Subword Tokens: ['The', 'ĠTrans', 'former', 'Ġarchitecture', 'Ġis', 'Ġunbelievably', 'Ġpowerful', '.']
------------------------------
Input IDs: [464, 3602, 16354, 10959, 318, 48943, 3665, 13]
------------------------------
Decoded Sentence: The Transformer architecture is unbelievably powerful.


## 2. Hands-on Prompting Techniques

This section corresponds to **Part 5** of the presentation.

Now that we have our API client ready, we can explore how to "program" the model using different prompting styles. We'll be using a modern chat model (`gpt-4o`), which has been fine-tuned to be excellent at following instructions.

To make our code cleaner, let's first create a small helper function to handle the API calls.

In [8]:
import openai

# Initialize the OpenAI client
# Make sure you have set your OpenAI API key in the environment variable OPENAI_API_KEY
client = openai.OpenAI()


def get_response(prompt, model="gpt-4o"):
    """
    A helper function to get a response from the OpenAI API.
    """     
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                # We can add a system message to set the model's behavior
                {"role": "system", "content": "You are a helpful assistant."},
                # The user's prompt
                {"role": "user", "content": prompt}
            ],
            temperature=0.7, # A value between 0 and 2. Higher values make the output more random.
            max_tokens=256   # The maximum number of tokens to generate.
        )
        return response.choices[0].message.content
    except Exception as e:
        # Handle potential API errors, e.g., invalid key
        return f"An error occurred: {e}"

# Let's test the helper function
test_prompt = "Hello, world!"
test_response = get_response(test_prompt)
print(f"Test Response: {test_response}")

Test Response: Hello! How can I assist you today?


### 2.1 Zero-Shot Prompting

This is the most basic form of prompting. We simply give the model a direct instruction without any prior examples.

Let's try a simple sentiment classification task.

In [9]:
zero_shot_prompt = """
Classify the following movie review as either 'Positive' or 'Negative'.

Review: 'This movie was a complete masterpiece! The acting was incredible and the story was unforgettable.'
"""

response = get_response(zero_shot_prompt)

print("--- Zero-Shot Classification ---")
print(f"Prompt:\n{zero_shot_prompt}")
print(f"\nModel Response:\n{response}")

--- Zero-Shot Classification ---
Prompt:

Classify the following movie review as either 'Positive' or 'Negative'.

Review: 'This movie was a complete masterpiece! The acting was incredible and the story was unforgettable.'


Model Response:
Positive


### 2.2 Few-Shot Prompting

For more complex tasks or when we need a very specific output format, we can provide a few examples ("shots") within the prompt itself. This is called in-context learning.

Here, we'll ask the model to extract specific data from a sentence and format it as `Attribute: Value`.

In [10]:
few_shot_prompt = """
Extract the Technology, Company, and Year from the following sentences.

Sentence: "In 1991, Guido van Rossum created Python."
Technology: Python, Company: N/A, Year: 1991
###
Sentence: "The first iPhone was released by Apple in 2007."
Technology: iPhone, Company: Apple, Year: 2007
###
Sentence: "In 2023, OpenAI released GPT-4."
Technology:
"""

response = get_response(few_shot_prompt)

print("--- Few-Shot Extraction ---")
print(f"Prompt:\n{few_shot_prompt}")
print(f"\nModel Response:\n{response}")

--- Few-Shot Extraction ---
Prompt:

Extract the Technology, Company, and Year from the following sentences.

Sentence: "In 1991, Guido van Rossum created Python."
Technology: Python, Company: N/A, Year: 1991
###
Sentence: "The first iPhone was released by Apple in 2007."
Technology: iPhone, Company: Apple, Year: 2007
###
Sentence: "In 2023, OpenAI released GPT-4."
Technology:


Model Response:
GPT-4, Company: OpenAI, Year: 2023


### 2.3 Chain-of-Thought (CoT) Prompting

When a problem requires multiple steps to solve, we can encourage the model to "think step by step." This often leads to more accurate results as it forces the model to work through its reasoning process before giving a final answer.

Let's try a simple word problem.

In [11]:
cot_prompt = """
Q: A grocery store has 5 boxes of apples, with 12 apples in each box. They sell 20 apples. Then, they receive a new shipment of 3 boxes, each containing 10 apples. How many apples do they have now?

Let's think step by step to find the answer.
"""

response = get_response(cot_prompt)

print("--- Chain-of-Thought Reasoning ---")
print(f"Prompt:\n{cot_prompt}")
print(f"\nModel Response:\n{response}")

--- Chain-of-Thought Reasoning ---
Prompt:

Q: A grocery store has 5 boxes of apples, with 12 apples in each box. They sell 20 apples. Then, they receive a new shipment of 3 boxes, each containing 10 apples. How many apples do they have now?

Let's think step by step to find the answer.


Model Response:
To solve this problem, we can break it down into steps:

1. **Calculate the initial number of apples:**
   - The grocery store has 5 boxes of apples, with 12 apples in each box.
   - Total apples = 5 boxes * 12 apples/box = 60 apples.

2. **Subtract the apples that are sold:**
   - They sell 20 apples.
   - Remaining apples = 60 apples - 20 apples = 40 apples.

3. **Add the apples from the new shipment:**
   - They receive a new shipment of 3 boxes, each containing 10 apples.
   - Apples in the new shipment = 3 boxes * 10 apples/box = 30 apples.

4. **Calculate the total number of apples after the new shipment:**
   - Total apples now = Remaining apples + Apples in the new shipment = 4

## 3. Reliable Outputs: Structured Data and Tool Use

This section corresponds to **Part 6** of the presentation.

While natural language is great for chat, applications need reliable, machine-readable data. Here we'll explore two powerful techniques for getting structured output.

### 3.1 JSON Mode

Modern APIs can guarantee that the model's output will be a syntactically correct JSON object. This is incredibly useful for preventing errors and simplifying your code.

In [12]:
import json

# Check if the client is initialized
if 'client' not in globals():
    print("OpenAI client not initialized. Please run the API key setup cell.")
else:
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            # This is the key parameter to enable JSON mode
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
                {"role": "user", "content": "Extract the user's name, age, and city from this sentence: 'Amanda, who is 34, just moved to sunny San Diego.'"}
            ]
        )

        output_str = response.choices[0].message.content
        print("--- Raw Model Output (a guaranteed JSON string) ---")
        print(output_str)

        # Because the output is guaranteed to be valid JSON, we can parse it directly
        parsed_json = json.loads(output_str)

        print("\n--- Parsed JSON Object ---")
        print(parsed_json)
        print(f"\nUser's Name: {parsed_json.get('name')}")

    except Exception as e:
        print(f"An error occurred: {e}")

--- Raw Model Output (a guaranteed JSON string) ---
{
  "name": "Amanda",
  "age": 34,
  "city": "San Diego"
}

--- Parsed JSON Object ---
{'name': 'Amanda', 'age': 34, 'city': 'San Diego'}

User's Name: Amanda


### 3.2 Tool Use / Function Calling

This is the full implementation of the ReAct principle. We give the LLM a set of "tools" (our Python functions) that it can ask our application to run. This allows the model to access real-time data or perform actions.

Let's walk through the full loop: `Define -> Request -> Execute -> Synthesize`.

In [13]:
import json

# Check if the client is initialized
if 'client' not in globals():
    print("OpenAI client not initialized. Please run the API key setup cell.")
else:
    # Step 1: Define a local Python function that the model can "call"
    def get_mock_stock_price(ticker_symbol: str):
        """Gets the mock stock price for a given ticker symbol."""
        print(f"--- Application: Running get_mock_stock_price for {ticker_symbol} ---")
        if "NVDA" in ticker_symbol.upper():
            return json.dumps({"ticker": "NVDA", "price": "125.50 USD"})
        elif "GOOG" in ticker_symbol.upper():
            return json.dumps({"ticker": "GOOG", "price": "178.20 USD"})
        else:
            return json.dumps({"ticker": ticker_symbol, "price": "unknown"})

    # Step 2: Define the "tool" specification for the API call
    tools = [
        {
            "type": "function",
            "function": {
                "name": "get_mock_stock_price",
                "description": "Get the current stock price for a specific ticker symbol",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "ticker_symbol": {
                            "type": "string",
                            "description": "The stock ticker symbol, e.g., 'NVDA'",
                        },
                    },
                    "required": ["ticker_symbol"],
                },
            },
        }
    ]
    
    # We will use this list to keep track of the conversation
    messages = [{"role": "user", "content": "What is the current stock price of NVIDIA (NVDA)?"}]

    # Step 3: Make the first call to the model
    try:
        print("--- Step 3: Sending user prompt to the model ---")
        first_response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        
        response_message = first_response.choices[0].message
        
        # Step 4: Check if the model wants to call a tool
        if response_message.tool_calls:
            print("--- Step 4: Model decided to call a tool ---")
            
            # Append the assistant's response to the message history
            messages.append(response_message)
            
            # Execute the function(s)
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                if function_name == "get_mock_stock_price":
                    function_args = json.loads(tool_call.function.arguments)
                    function_response = get_mock_stock_price(
                        ticker_symbol=function_args.get("ticker_symbol")
                    )
                    
                    # Append the function's response to the message history
                    messages.append(
                        {
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "name": function_name,
                            "content": function_response,
                        }
                    )
            
            # Step 5: Make the second call to the model with the tool's response
            print("--- Step 5: Sending tool response back to the model for synthesis ---")
            second_response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
            )
            
            print("\n--- Final Answer ---")
            print(second_response.choices[0].message.content)

        else:
            print("--- Model responded directly ---")
            print(response_message.content)

    except Exception as e:
        print(f"An error occurred: {e}")

--- Step 3: Sending user prompt to the model ---
--- Step 4: Model decided to call a tool ---
--- Application: Running get_mock_stock_price for NVDA ---
--- Step 5: Sending tool response back to the model for synthesis ---

--- Final Answer ---
The current stock price of NVIDIA (NVDA) is approximately 125.50 USD.


## 4. The LLM Ecosystem: API vs. Local

This final section corresponds to **Part 7** of the presentation. We'll look at the two primary ways to interact with models, which we've already set up: via a cloud API and by running a model locally on our own machine.

### 4.1 API Access (Recap)

All of the previous examples that used the `openai` client are examples of API access. We sent a request to a powerful model hosted on OpenAI's servers and received a response. This is the most common way to access state-of-the-art models without needing powerful hardware.

---

### 4.2 Local Models with Ollama

The second approach is to run an open-source model directly on your computer. This gives you complete privacy and control. We'll use **Ollama**, a fantastic tool that makes this process simple.

**Setup Instructions:**

1.  **Install Ollama:** If you haven't already, go to [ollama.com](https://ollama.com) and download the application for your operating system.
2.  **Run Ollama:** Make sure the Ollama application is running in the background.
3.  **Pull a Model:** Open your terminal (not in this notebook) and run the following command to download a small, efficient model. We'll use `gemma:2b`, a 2-billion parameter model from Google that is great for demonstrations.

```bash
ollama pull gemma:2b

In [14]:
import ollama

# This code assumes the Ollama application is running on your machine.
# If it is not, this cell will raise an error.

try:
    response = ollama.chat(
        # This model name must match one you have pulled with `ollama pull`
        model='gemma:2b',
        messages=[
            {'role': 'user', 'content': 'In one short sentence, why is learning about Transformers important?'},
        ],
    )
    
    print("--- Response from local 'gemma:2b' model ---")
    print(response['message']['content'])

except Exception as e:
    print("An error occurred. Is the Ollama application running on your computer?")
    print(f"Error details: {e}")

--- Response from local 'gemma:2b' model ---
Learning about Transformers is important because they are a groundbreaking artificial intelligence technology that has revolutionized natural language processing (NLP) and has numerous applications in various fields, including language modeling, machine translation, and text generation.


# Homework

https://huggingface.co/docs/transformers/tasks/sequence_classification